In [2]:
import requests
BASE_URL = "http://127.0.0.1:8021"

In [3]:
# POST /links/shorten
original_url = "https://www.google.com"
params = {"original_url": original_url}

response = requests.post(f"{BASE_URL}/links/shorten", params=params)
print(response.status_code)
print(response.json())

short_url = response.json()["short_url"]
short_code = short_url.split("/")[-1]

200
{'short_url': 'http://127.0.0.1:8021/4uOoUD'}


In [4]:
# POST /links/shorten с alias
original_url = "https://www.google.com"
custom_alias = "ggl1"
params = {
    "original_url": original_url,
    "custom_alias": custom_alias,
}

response = requests.post(f"{BASE_URL}/links/shorten", params=params)
print("POST /links/shorten", response.status_code)
print(response.json())

short_url = response.json()["short_url"]
short_code = short_url.split("/")[-1]

POST /links/shorten 200
{'short_url': 'http://127.0.0.1:8021/ggl1'}


In [5]:
# POST /links/shorten с alias и временем умирания
original_url = "https://www.google.com"
custom_alias = "ggl2"
params = {
    "original_url": original_url,
    "custom_alias": custom_alias,
    "expires_at": "2026-03-20T12:00:00"
}

response = requests.post(f"{BASE_URL}/links/shorten", params=params)
print("POST /links/shorten", response.status_code)
print(response.json())

short_url = response.json()["short_url"]
short_code = short_url.split("/")[-1]

POST /links/shorten 200
{'short_url': 'http://127.0.0.1:8021/ggl2'}


In [6]:
# GET /links/{short_code}
response = requests.get(f"{BASE_URL}/{short_code}", allow_redirects=False)
print(f"GET /{short_code} (редирект)")
print(response.status_code)
print("Location header:", response.headers.get("location"))

GET /ggl2 (редирект)
307
Location header: https://www.google.com


In [7]:
# Статистика по ссылке
response = requests.get(f"{BASE_URL}/links/{short_code}/stats")
print(f"GET /links/{short_code}/stats")
print(response.status_code)
print(response.json())

GET /links/ggl2/stats
200
{'original_url': 'https://www.google.com', 'created_at': '2026-03-15T20:37:13.537408', 'clicks': 1, 'last_used_at': '2026-03-15T20:37:13.565628'}


In [8]:
# PUT /links/{short_code}
new_url = "https://chance4life.org/"
params = {"original_url": new_url}
response = requests.put(f"{BASE_URL}/links/{short_code}", params=params)
print(f"PUT /links/{short_code}")
print(response.status_code)
print(response.json())

PUT /links/ggl2
200
{'message': 'Link updated'}


In [9]:
# DELETE /links/{short_code}
response = requests.delete(f"{BASE_URL}/links/{short_code}")
print(f"DELETE /links/{short_code}")
print(response.status_code)
print(response.json())

DELETE /links/ggl2
200
{'message': 'Link deleted'}


In [10]:
# GET /links/search
search_url = "https://www.google.com"
params = {"original_url": search_url}
response = requests.get(f"{BASE_URL}/links/search", params=params)
print("GET /links/search")
print(response.status_code)
print(response.json())

GET /links/search
200
[{'short_code': 'WxESoD', 'original_url': 'https://www.google.com'}, {'short_code': 'z0PehG', 'original_url': 'https://www.google.com'}, {'short_code': 'Dt9axr', 'original_url': 'https://www.google.com'}, {'short_code': 'ggl', 'original_url': 'https://www.google.com'}, {'short_code': 'usOze6', 'original_url': 'https://www.google.com'}, {'short_code': '4uOoUD', 'original_url': 'https://www.google.com'}, {'short_code': 'ggl1', 'original_url': 'https://www.google.com'}]


In [11]:
# Регистрация
user_data = {"email": "testuser3@example.com", "password": "123456"}
response = requests.post(f"{BASE_URL}/users/register", params=user_data)
print("POST /users/register", response.status_code)
print(response.json())

user_id = response.json().get("id")

POST /users/register 200
{'message': 'User registered', 'user_id': 3}


In [12]:
# Ссылка с пользователем
original_url = "https://lms.hse.ru/"
params = {"original_url": original_url, "user_id": user_id}
response = requests.post(f"{BASE_URL}/links/shorten", params=params)
print("POST /links/shorten (с user_id)")
print(response.status_code)
link_data = response.json()
print(link_data)

short_url = link_data["short_url"]
short_code = short_url.split("/")[-1]

POST /links/shorten (с user_id)
200
{'short_url': 'http://127.0.0.1:8021/W2puvO'}


In [13]:
# Статистика ссылки
response = requests.get(f"{BASE_URL}/links/{short_code}/stats")
print(f"GET /links/{short_code}/stats")
print(response.status_code)
print(response.json())

GET /links/W2puvO/stats
200
{'original_url': 'https://lms.hse.ru/', 'created_at': '2026-03-15T20:37:13.822719', 'clicks': 0, 'last_used_at': None}


In [14]:
# При новых обращениях кол-во кликов увеличивается
for _ in range(3):
    requests.get(f"{BASE_URL}/{short_code}", allow_redirects=False)

response = requests.get(f"{BASE_URL}/links/{short_code}/stats")
print(response.json())

{'original_url': 'https://lms.hse.ru/', 'created_at': '2026-03-15T20:37:13.822719', 'clicks': 3, 'last_used_at': '2026-03-15T20:37:13.840245'}
